In [21]:
import pandas as pd
import requests
import numpy as np
from bs4 import BeautifulSoup
import yfinance
import os
import pygsheets
from decimal import Decimal, ROUND_HALF_UP
import locale
from datetime import datetime, timedelta
import json
import datawrapper
import time
from datetime import datetime, timedelta

import io
#https://github.com/patrickloeber/python-github-action-template?tab=readme-ov-file
# Set the locale to English (United States)
locale.setlocale(locale.LC_TIME, 'en_US.UTF-8')
#authorization
if "PYGSHEET" in os.environ:

    AUTH_DATA = os.environ["PYGSHEET"]
    whatever_json=json.loads(AUTH_DATA, strict=False)
    credentials = whatever_json

    gc = pygsheets.authorize(service_account_json=(json.dumps(whatever_json))
    )
else:

    gc = pygsheets.authorize(service_file='/Users/halukamaier-borst/Documents/trumptracker-459020-5b428916efa8.json')



In [11]:

# Get today's date
today = datetime.now()

# Get date 365 days ago
date_365_days_ago = today - timedelta(days=365)

# Format as string if needed
date_365_days_ago_str = date_365_days_ago.strftime("%Y-%m-%d")

print(date_365_days_ago_str)
today

2025-01-06


datetime.datetime(2026, 1, 6, 9, 9, 6, 977326)

In [12]:

GoldPrice = yfinance.download (tickers = "GC=F", start = date_365_days_ago_str, 
                              interval = "1d")
GoldPrice

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
Date,,,,,
2025-01-06,2638.399902,2647.000000,2617.300049,2645.500000,960
2025-01-07,2656.699951,2657.500000,2653.000000,2653.399902,643
2025-01-08,2664.500000,2676.899902,2653.500000,2655.500000,999
2025-01-09,2683.800049,2686.300049,2667.899902,2669.699951,1139
2025-01-10,2708.500000,2720.100098,2683.699951,2686.100098,403
...,...,...,...,...,...
2025-12-30,4370.100098,4403.600098,4338.000000,4346.399902,1837
2025-12-31,4325.600098,4363.799805,4285.000000,4333.500000,785


In [13]:
msci_world = yfinance.download (tickers = "^990100-USD-STRD", start = date_365_days_ago_str, 
                              interval = "1d")
msci_world

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^990100-USD-STRD,^990100-USD-STRD,^990100-USD-STRD,^990100-USD-STRD,^990100-USD-STRD
Date,,,,,
2025-01-06,3763.199951,3785.000000,3736.100098,3738.600098,0
2025-01-07,3733.600098,3780.300049,3723.399902,3763.500000,0
2025-01-08,3732.199951,3737.399902,3710.899902,3731.699951,0
2025-01-09,3732.100098,3733.600098,3725.699951,3732.300049,0
2025-01-10,3678.199951,3731.800049,3668.100098,3731.699951,0
...,...,...,...,...,...
2025-12-30,4458.299805,4468.100098,4456.299805,4461.299805,0
2025-12-31,4430.399902,4457.700195,4430.399902,4457.600098,0


In [14]:
bitcoin = yfinance.download (tickers = "BTC-USD", start = date_365_days_ago_str, 
                              interval = "1d")
bitcoin

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2025-01-06,102078.085938,102482.875000,97926.148438,98314.953125,51823432705
2025-01-07,96922.703125,102712.484375,96132.875000,102248.851562,58685738547
2025-01-08,95043.523438,97258.320312,92525.843750,96924.164062,63875859171
2025-01-09,92484.039062,95349.718750,91220.843750,95043.484375,62777261693
2025-01-10,94701.453125,95770.609375,92250.093750,92494.492188,62058693684
...,...,...,...,...,...
2026-01-02,89944.695312,90884.460938,88298.617188,88733.062500,46398906171
2026-01-03,90603.187500,90679.570312,89328.070312,89945.054688,20774828592


In [27]:
def reduce_index(df,indexname):
    index_red=pd.DataFrame()
    df["Date"]=df.index
    index_red["Date"]=df["Date"]

    index_red[indexname]=df[df.columns[0]]

    index_red=index_red.reset_index(drop=True)
    return index_red


In [28]:
gold_red=reduce_index(GoldPrice,"GOLD")
msci_world_red=reduce_index(msci_world,"MSCI_WORLD")
bitcoin_red=reduce_index(bitcoin,"BITCOIN")
bitcoin_red

,Date,BITCOIN
0,2025-01-06,102078.085938
1,2025-01-07,96922.703125
2,2025-01-08,95043.523438
3,2025-01-09,92484.039062
4,2025-01-10,94701.453125
...,...,...
361,2026-01-02,89944.695312
362,2026-01-03,90603.187500
363,2026-01-04,91413.492188
364,2026-01-05,93882.554688


In [29]:
gold_msci=pd.merge(gold_red, msci_world_red, on="Date", how="outer")
gold_msci=gold_msci.fillna(method="ffill")
gold_msci

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/276846787.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gold_msci=gold_msci.fillna(method="ffill")


,Date,GOLD,MSCI_WORLD
0,2025-01-06,2638.399902,3763.199951
1,2025-01-07,2656.699951,3733.600098
2,2025-01-08,2664.500000,3732.199951
3,2025-01-09,2683.800049,3732.100098
4,2025-01-10,2708.500000,3678.199951
...,...,...,...
250,2025-12-30,4370.100098,4458.299805
251,2025-12-31,4325.600098,4430.399902
252,2026-01-02,4314.399902,4445.200195
253,2026-01-05,4436.899902,4477.899902


In [30]:
gold_msci_bitcoin=pd.merge(gold_msci, bitcoin_red, on="Date", how="outer")
gold_msci_bitcoin=gold_msci_bitcoin.fillna(method="ffill")
today_str=today.strftime("%Y-%m-%d")

gold_msci_bitcoin_without_today=gold_msci_bitcoin[gold_msci_bitcoin["Date"]!=today_str]
gold_msci_bitcoin_without_today=gold_msci_bitcoin_without_today.dropna()
gold_msci_bitcoin_without_today

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/3567577943.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  gold_msci_bitcoin=gold_msci_bitcoin.fillna(method="ffill")


,Date,GOLD,MSCI_WORLD,BITCOIN
0,2025-01-06,2638.399902,3763.199951,102078.085938
1,2025-01-07,2656.699951,3733.600098,96922.703125
2,2025-01-08,2664.500000,3732.199951,95043.523438
3,2025-01-09,2683.800049,3732.100098,92484.039062
4,2025-01-10,2708.500000,3678.199951,94701.453125
...,...,...,...,...
360,2026-01-01,4325.600098,4430.399902,88731.984375
361,2026-01-02,4314.399902,4445.200195,89944.695312
362,2026-01-03,4314.399902,4445.200195,90603.187500
363,2026-01-04,4314.399902,4445.200195,91413.492188


In [31]:
for col in gold_msci_bitcoin_without_today.columns[1:]:
    gold_msci_bitcoin_without_today[col]= gold_msci_bitcoin_without_today[col].apply(lambda x: Decimal(str(x)))

    # Example: Perform precise rounding on the Decimal column
    gold_msci_bitcoin_without_today[col]= gold_msci_bitcoin_without_today[col].apply(lambda x: x.quantize(Decimal('1'), rounding=ROUND_HALF_UP))
gold_msci_bitcoin_without_today

,Date,GOLD,MSCI_WORLD,BITCOIN
0,2025-01-06,2638,3763,102078
1,2025-01-07,2657,3734,96923
2,2025-01-08,2665,3732,95044
3,2025-01-09,2684,3732,92484
4,2025-01-10,2709,3678,94701
...,...,...,...,...
360,2026-01-01,4326,4430,88732
361,2026-01-02,4314,4445,89945
362,2026-01-03,4314,4445,90603
363,2026-01-04,4314,4445,91413


In [32]:
gold_msci_bitcoin_without_today.to_csv("sofar.csv", index=False)

In [33]:
def retrieve_from_investing(index):
    index_to_shortform_dict={"MSCI_WORLD":"indices/msci-world-historical-data","GOLD":"commodities/gold-historical-data","BITCOIN":"crypto/bitcoin/historical-data"}
    now = datetime.now()
    yesterday = now - timedelta(days=1)
    formatted_date = yesterday.strftime("%m/%d/%Y")
    index_shortform=index_to_shortform_dict[index]


    url=f"https://www.investing.com/{index_shortform}#:~:text=Daily-,Download,-{formatted_date}"
    print(url)
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.102 Safari/537.36'}
    response = requests.get(url, headers=headers)



    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content of the page
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the first table on the page
        table = soup.find('table', class_="freeze-column-w-1 w-full overflow-x-auto text-xs leading-4")

        # Extract rows from the table
        rows = table.find_all('tr')

        # Extract data from each row
        table_data = []
        for row in rows:
            cells = row.find_all(['td', 'th'])  # Get all cells (both <td> and <th>)
            cell_data = [cell.get_text(strip=True) for cell in cells]  # Extract text
            table_data.append(cell_data)

        # Return the extracted table data
        return(table_data)
    else:
        print(f"Failed to retrieve the page. Status code: {response.status_code}")

In [34]:
filepath = "https://www.cryptodatadownload.com/cdd/Binance_BTCUSDT_d.csv"


# Now we want to create a dataframe and use Pandas' to_csv function to read in our file
df = pd.read_csv(filepath, skiprows=1)  # we use skiprows parameter because first row contains our web address

# Now that we have loaded our data into the dataframe, we can preview it using the print & .head() function
df_red=df[["Date","Close"]]
df_red["Date"]=pd.to_datetime(df_red["Date"])
df_red["Close"]=df_red["Close"].apply(lambda x: Decimal(str(x)))
df_red=df_red.dropna(how="all")
crosscheck_data_red=df_red
crosscheck_data_red=crosscheck_data_red.dropna(subset=["Date"])
crosscheck_data_red.columns=["Date","Bitcoin"+"check"]
crosscheck_data_red

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/185079238.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_red["Date"]=pd.to_datetime(df_red["Date"])
/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/185079238.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_red["Close"]=df_red["Close"].apply(lambda x: Decimal(str(x)))


,Date,Bitcoincheck
0,2026-01-04,91529.73
1,2026-01-03,90628.01
2,2026-01-02,89995.13
3,2026-01-01,88839.04
4,2025-12-31,87648.22
...,...,...
3058,2017-08-21,4016.0
3059,2017-08-20,4086.29
3060,2017-08-19,4139.98
3061,2017-08-18,4108.37


In [35]:
df_to_check=gold_msci_bitcoin_without_today.copy()
columns_to_round = ['GOLD','MSCI_WORLD','BITCOIN']
    
# Apply Decimal rounding to the specified columns
df_to_check[columns_to_round] = df_to_check[columns_to_round].applymap(lambda x: Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

for index_name in gold_msci_bitcoin_without_today.columns[1:]:
    print(index_name)
    if index_name!="BITCOIN":
        array=retrieve_from_investing(index_name)
        crosscheck_data=pd.DataFrame(array)
        crosscheck_data.replace("",np.nan,inplace=True)
        crosscheck_data=crosscheck_data.dropna(how="all")
        crosscheck_data[0]=pd.to_datetime(crosscheck_data[0], format="%b %d, %Y", errors="coerce")
        crosscheck_data_red=crosscheck_data[[0,1]]
        crosscheck_data_red=crosscheck_data_red.dropna(subset=[0])
        crosscheck_data_red.columns=["Date",index_name+"check"]
        crosscheck_data_red=crosscheck_data_red.sort_values(by="Date", ascending=True)
        crosscheck_data_red[index_name+"check"]=crosscheck_data_red[index_name+"check"].str.replace(",","")
        crosscheck_data_red[index_name+"check"]=crosscheck_data_red[index_name+"check"].astype(float)
        crosscheck_data_red[index_name+"check"]= crosscheck_data_red[index_name+"check"].apply(lambda x: Decimal(str(x)))
    else:
        filepath = "https://www.cryptodatadownload.com/cdd/Binance_BTCUSDT_d.csv"


        # Now we want to create a dataframe and use Pandas' to_csv function to read in our file
        df = pd.read_csv(filepath, skiprows=1)  # we use skiprows parameter because first row contains our web address

        # Now that we have loaded our data into the dataframe, we can preview it using the print & .head() function
        df_red=df[["Date","Close"]]
        df_red["Date"]=pd.to_datetime(df_red["Date"])
        df_red["Close"]=df_red["Close"].apply(lambda x: Decimal(str(x)))
        df_red=df_red.dropna(how="all")
        crosscheck_data_red=df_red
        crosscheck_data_red=crosscheck_data_red.dropna(subset=["Date"])
        crosscheck_data_red.columns=["Date","BITCOINcheck"]
        crosscheck_data_red
    # Example: Perform precise rounding on the Decimal column
    crosscheck_data_red[index_name+"check"] = crosscheck_data_red[index_name+"check"].apply(lambda x: x.quantize(Decimal('1'), rounding=ROUND_HALF_UP))
    print("crosscheckdata")
    print(crosscheck_data_red.tail(5))
    print("df_to_check")

    print(df_to_check.tail(5))

#    df_to_check=pd.merge(df_to_check,crosscheck_data_red, on="Date", how="outer", suffixes=["_raw","_crosscheck"])

    df_to_check=pd.merge(df_to_check,crosscheck_data_red, on="Date", how="outer", suffixes=["_raw","_crosscheck"])
df_to_check=df_to_check.fillna(method="ffill")
df_to_check


/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/384376811.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_to_check[columns_to_round] = df_to_check[columns_to_round].applymap(lambda x: Decimal(str(x)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))


GOLD
https://www.investing.com/commodities/gold-historical-data#:~:text=Daily-,Download,-01/05/2026
crosscheckdata
        Date GOLDcheck
5 2026-01-01      4362
4 2026-01-02      4330
3 2026-01-04      4382
2 2026-01-05      4452
1 2026-01-06      4477
df_to_check
          Date  GOLD MSCI_WORLD BITCOIN
360 2026-01-01  4326       4430   88732
361 2026-01-02  4314       4445   89945
362 2026-01-03  4314       4445   90603
363 2026-01-04  4314       4445   91413
364 2026-01-05  4437       4478   93883
MSCI_WORLD
https://www.investing.com/indices/msci-world-historical-data#:~:text=Daily-,Download,-01/05/2026
crosscheckdata
        Date MSCI_WORLDcheck
5 2026-01-01            4431
4 2026-01-02            4445
3 2026-01-04            4441
2 2026-01-05            4478
1 2026-01-06            4485
df_to_check
          Date  GOLD MSCI_WORLD BITCOIN GOLDcheck
361 2026-01-02  4314       4445   89945      4330
362 2026-01-03  4314       4445   90603       NaN
363 2026-01-04  4314       4445   91

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/384376811.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_red["Date"]=pd.to_datetime(df_red["Date"])
/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/384376811.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_red["Close"]=df_red["Close"].apply(lambda x: Decimal(str(x)))
/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/384376811.py:49: FutureWarning: DataFrame.fillna with 'method' is deprecated and

,Date,GOLD,MSCI_WORLD,BITCOIN,GOLDcheck,MSCI_WORLDcheck,BITCOINcheck
0,2017-08-17,NaN,NaN,NaN,NaN,NaN,4285
1,2017-08-18,NaN,NaN,NaN,NaN,NaN,4108
2,2017-08-19,NaN,NaN,NaN,NaN,NaN,4140
3,2017-08-20,NaN,NaN,NaN,NaN,NaN,4086
4,2017-08-21,NaN,NaN,NaN,NaN,NaN,4016
...,...,...,...,...,...,...,...
3060,2026-01-02,4314,4445,89945,4330,4445,89995
3061,2026-01-03,4314,4445,90603,4330,4445,90628
3062,2026-01-04,4314,4445,91413,4382,4441,91530
3063,2026-01-05,4437,4478,93883,4452,4478,91530


In [36]:
for index_name in gold_msci_bitcoin_without_today.columns[1:]:

    df_to_check[index_name+"overlap"]=df_to_check[index_name]/df_to_check[index_name+"check"]
df_to_check

,Date,GOLD,MSCI_WORLD,BITCOIN,GOLDcheck,MSCI_WORLDcheck,BITCOINcheck,GOLDoverlap,MSCI_WORLDoverlap,BITCOINoverlap
0,2017-08-17,NaN,NaN,NaN,NaN,NaN,4285,NaN,NaN,NaN
1,2017-08-18,NaN,NaN,NaN,NaN,NaN,4108,NaN,NaN,NaN
2,2017-08-19,NaN,NaN,NaN,NaN,NaN,4140,NaN,NaN,NaN
3,2017-08-20,NaN,NaN,NaN,NaN,NaN,4086,NaN,NaN,NaN
4,2017-08-21,NaN,NaN,NaN,NaN,NaN,4016,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3060,2026-01-02,4314,4445,89945,4330,4445,89995,0.9963048498845265588914549654,1,0.9994444135785321406744819157
3061,2026-01-03,4314,4445,90603,4330,4445,90628,0.9963048498845265588914549654,1,0.9997241470627179238204528402
3062,2026-01-04,4314,4445,91413,4382,4441,91530,0.9844819717024189867640346874,1.000900698040981760864670119,0.9987217305801376597836774828
3063,2026-01-05,4437,4478,93883,4452,4478,91530,0.9966307277628032345013477089,1,1.025707418332787064350486179


In [37]:
overlap_cols=df_to_check.columns[df_to_check.columns.str.contains("overlap")]
overlap_cols

Index(['GOLDoverlap', 'MSCI_WORLDoverlap', 'BITCOINoverlap'], dtype='object')

In [38]:
df_enough_overlap=df_to_check[((df_to_check[overlap_cols]>0.99) & (df_to_check[overlap_cols]<1.01)).all(axis=1)]
df_enough_overlap

,Date,GOLD,MSCI_WORLD,BITCOIN,GOLDcheck,MSCI_WORLDcheck,BITCOINcheck,GOLDoverlap,MSCI_WORLDoverlap,BITCOINoverlap
3035,2025-12-08,4187,4405,90640,4218,4405,90634,0.9926505452821242294926505453,1,1.000066200322174901251186089
3036,2025-12-09,4207,4400,92692,4236,4400,92679,0.9931539187913125590179414542,1,1.000140269100875063390843665
3037,2025-12-10,4196,4425,92021,4225,4425,92015,0.9931360946745562130177514793,1,1.000065206759767429223496169
3038,2025-12-11,4286,4444,92511,4313,4444,92513,0.9937398562485508926501275214,1,0.9999783814166657658923610736
3039,2025-12-12,4300,4408,90270,4328,4408,90268,0.9935304990757855822550831793,1,1.000022156245845703903930518
3040,2025-12-13,4300,4408,90299,4328,4408,90240,0.9935304990757855822550831793,1,1.000653812056737588652482270
3041,2025-12-14,4300,4408,88175,4328,4408,88172,0.9935304990757855822550831793,1,1.000034024406841174068865399
3042,2025-12-15,4307,4409,86420,4335,4409,86432,0.9935409457900807381776239908,1,0.9998611625323954091077378749
3043,2025-12-16,4305,4393,87844,4332,4393,87863,0.9937673130193905817174515235,1,0.9997837542537814552200585002
3044,2025-12-17,4348,4352,86144,4374,4352,86243,0.9940557841792409693644261545,1,0.9988520807485824936516586853


In [39]:
checked_df_red=df_enough_overlap[df_enough_overlap.columns[:4]]
checked_df_red["Date"]=checked_df_red["Date"].dt.strftime("%Y-%m-%d")
checked_df_red

/var/folders/d_/6z_vk0mx7j10g1h2zrjh_6f00000gn/T/ipykernel_34346/238757726.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  checked_df_red["Date"]=checked_df_red["Date"].dt.strftime("%Y-%m-%d")


,Date,GOLD,MSCI_WORLD,BITCOIN
3035,2025-12-08,4187,4405,90640
3036,2025-12-09,4207,4400,92692
3037,2025-12-10,4196,4425,92021
3038,2025-12-11,4286,4444,92511
3039,2025-12-12,4300,4408,90270
3040,2025-12-13,4300,4408,90299
3041,2025-12-14,4300,4408,88175
3042,2025-12-15,4307,4409,86420
3043,2025-12-16,4305,4393,87844
3044,2025-12-17,4348,4352,86144


In [19]:
#Open the google spreadsheet
sh = gc.open('TrumptrackerRaw')
sheet_name = "Assets"  # Replace with the name of the sheet you want to select
wks = sh.worksheet_by_title(sheet_name)


#Get the data from the Sheet into python as DF
read_indices = wks.get_as_df()
read_indices_red=read_indices[read_indices.columns[:4]]
read_indices_red

,Date,GOLD,MSCI_WORLD,BITCOIN
0,2024-12-11,2734,3842,101173
1,2024-12-12,2688,3826,100043
2,2024-12-13,2656,3817,101459
3,2024-12-14,2656,3817,101373
4,2024-12-15,2656,3817,104299
...,...,...,...,...
359,2025-12-05,4213,4419,89388
360,2025-12-06,4213,4419,89272
361,2025-12-07,4213,4419,90406
362,2025-12-08,4187,4405,90640


In [20]:
read_plus_new_checked=pd.concat([read_indices_red,checked_df_red], axis=0, ignore_index=True)

read_plus_new_checked.drop_duplicates(subset=["Date"], keep="first", inplace=True)
read_plus_new_checked.reset_index(drop=True, inplace=True)
read_plus_new_checked

,Date,GOLD,MSCI_WORLD,BITCOIN
0,2024-12-11,2734,3842,101173
1,2024-12-12,2688,3826,100043
2,2024-12-13,2656,3817,101459
3,2024-12-14,2656,3817,101373
4,2024-12-15,2656,3817,104299
...,...,...,...,...
360,2025-12-06,4213,4419,89272
361,2025-12-07,4213,4419,90406
362,2025-12-08,4187,4405,90640
363,2025-12-09,4207,4400,92692


In [21]:
read_plus_new_checked["Date"]=pd.to_datetime(read_plus_new_checked["Date"])
read_plus_new_checked=read_plus_new_checked[read_plus_new_checked["Date"]>date_365_days_ago]
read_plus_new_checked.sort_values(by="Date", ascending=True, inplace=True)
read_plus_new_checked

,Date,GOLD,MSCI_WORLD,BITCOIN
0,2024-12-11,2734,3842,101173
1,2024-12-12,2688,3826,100043
2,2024-12-13,2656,3817,101459
3,2024-12-14,2656,3817,101373
4,2024-12-15,2656,3817,104299
...,...,...,...,...
360,2025-12-06,4213,4419,89272
361,2025-12-07,4213,4419,90406
362,2025-12-08,4187,4405,90640
363,2025-12-09,4207,4400,92692


In [22]:
def converter_perc(index_name):
    df_perc=pd.DataFrame()
    df_perc["Date"]=read_plus_new_checked["Date"]
    col_important=read_plus_new_checked[index_name]
    base_value=col_important.iloc[0]
    df_perc[index_name+"_perc"]=(col_important-base_value)*100/base_value
    df_perc[index_name+"_perc"] = df_perc[index_name+"_perc"].apply(lambda x: float(Decimal(str(x)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)))

    return(df_perc)

In [23]:
for col in read_plus_new_checked.columns[1:]:
    df_perc=converter_perc(col)
    read_plus_new_checked=pd.merge(read_plus_new_checked,df_perc,on="Date",how="outer",suffixes=["_abs","_perc"])
read_plus_new_checked 

,Date,GOLD,MSCI_WORLD,BITCOIN,GOLD_perc,MSCI_WORLD_perc,BITCOIN_perc
0,2024-12-11,2734,3842,101173,0.00,0.00,0.00
1,2024-12-12,2688,3826,100043,-1.68,-0.42,-1.12
2,2024-12-13,2656,3817,101459,-2.85,-0.65,0.28
3,2024-12-14,2656,3817,101373,-2.85,-0.65,0.20
4,2024-12-15,2656,3817,104299,-2.85,-0.65,3.09
...,...,...,...,...,...,...,...
360,2025-12-06,4213,4419,89272,54.10,15.02,-11.76
361,2025-12-07,4213,4419,90406,54.10,15.02,-10.64
362,2025-12-08,4187,4405,90640,53.15,14.65,-10.41
363,2025-12-09,4207,4400,92692,53.88,14.52,-8.38


In [24]:



#open the google spreadsheet (where 'PY to Gsheet Test' is the name of my sheet)
sh = gc.open('TrumptrackerRaw')

sheet_name = "Assets"  # Replace with the name of the sheet you want to select
wks = sh.worksheet_by_title(sheet_name)

wks.set_dataframe(read_plus_new_checked,(0,0))


In [25]:
if "DWKEY" in os.environ:

    DWKEY = os.environ["DWKEY"]
    print(len(DWKEY))


else:
    data=pd.read_csv("/Users/halukamaier-borst/Documents/Keys.csv")
    DWKEY=data.APIKey.iloc[1]

In [28]:
dw_connection = datawrapper.Datawrapper(access_token = DWKEY)
allcharts=dw_connection.get_charts(limit=500)

dw_connection.publish_chart(chart_id="NH3ER")

{'data': {'publicId': 'NH3ER',
  'language': 'de-DE',
  'theme': 'handelsblatt',
  'id': 'NH3ER',
  'type': 'd3-lines',
  'title': 'So haben sich verschiedene Assets in den letzten 365 Tagen verändert',
  'lastEditStep': 5,
  'publishedAt': '2025-12-10T08:46:12.670Z',
  'publicUrl': 'https://datawrapper.dwcdn.net/NH3ER/4/',
  'publicVersion': 4,
  'deleted': False,
  'forkable': False,
  'isFork': False,
  'metadata': {'data': {'changes': [],
    'transpose': False,
    'vertical-header': True,
    'horizontal-header': True,
    'column-format': {'GOLD': {'type': 'auto', 'ignore': True},
     'DAX40': {'type': 'auto', 'ignore': True},
     'S&P500': {'type': 'auto', 'ignore': True},
     'BITCOIN': {'type': 'auto', 'ignore': True},
     'SHANGHAI': {'type': 'auto', 'ignore': True},
     'DAX40_abs': {'type': 'auto', 'ignore': True},
     'GOLD_perc': {'type': 'auto', 'number-append': '%', 'number-prepend': ''},
     'DAX40_perc': {'type': 'auto',
      'number-append': ' %',
      'num